<h1>Chapter 4 - Preparing Data for Vector Stores</h1>
<i>Prepare and chunk text data to store them in a vector store.</i>

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/"><img src="https://img.shields.io/badge/O'Reilly-white.svg?logo=data:image/svg%2bxml;base64,PHN2ZyB3aWR0aD0iMzQiIGhlaWdodD0iMjciIHZpZXdCb3g9IjAgMCAzNCAyNyIgZmlsbD0ibm9uZSIgeG1sbnM9Imh0dHA6Ly93d3cudzMub3JnLzIwMDAvc3ZnIj4KPGNpcmNsZSBjeD0iMTMiIGN5PSIxNCIgcj0iMTEiIHN0cm9rZT0iI0Q0MDEwMSIgc3Ryb2tlLXdpZHRoPSI0Ii8+CjxjaXJjbGUgY3g9IjMwLjUiIGN5PSIzLjUiIHI9IjMuNSIgZmlsbD0iI0Q0MDEwMSIvPgo8L3N2Zz4K"></a>
<a href="https://github.com/polzerdo55862/RAG-with-Python-Cookbook"><img src="https://img.shields.io/badge/GitHub%20Repository-black?logo=github"></a>
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/polzerdo55862/RAG-with-Python-Cookbook/blob/main/ch04_data_preparation_chunking_data/chunking_data.ipynb)

---

This notebook is for Chapter 4 of the [RAG with Python Cookbook](https://learning.oreilly.com/library/view/rag-with-python/9798341600553/) book by [Dominik Polzer](https://www.linkedin.com/in/polzerdo/).

---

<a href="https://learning.oreilly.com/library/view/rag-with-python/9798341600553/">
  <img src="https://raw.githubusercontent.com/polzerdo55862/RAG-with-Python-Cookbook/main/rag_cookbook.png" width="350" />
</a>


## Prerequisites

If you are viewing this notebook on Google Colab (or any other cloud vendor), you need to uncomment and run the following codeblock to install the dependencies for this chapter.

In [1]:
!pip install PyPDF2==3.0.1
!pip install pandas==2.2.3
!pip install pydantic==2.11.5
!pip install openai==1.83.0
!pip install matplotlib==3.10.3
!pip install scikit-learn==1.6.1
!pip install python-docx==1.1.2
!pip install nltk==3.9.1
!pip install langchain==0.3.25
!pip install langchain_openai==0.3.21
!pip install langchain-experimental==0.3.4
!pip install python-dotenv==1.0.0

  Using cached langchain-0.3.25-py3-none-any.whl.metadata (7.8 kB)
Using cached langchain-0.3.25-py3-none-any.whl (1.0 MB)
  Attempting uninstall: langchain
    Found existing installation: langchain 0.3.27
    Uninstalling langchain-0.3.27:
      Successfully uninstalled langchain-0.3.27
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.3.31 requires langchain<2.0.0,>=0.3.27, but you have langchain 0.3.25 which is incompatible.
  Using cached langchain-1.2.10-py3-none-any.whl.metadata (5.7 kB)
INFO: pip is looking at multiple versions of langchain to determine which version is compatible with other requirements. This could take a while.
  Using cached langchain-1.2.9-py3-none-any.whl.metadata (5.7 kB)
  Using cached langchain-1.2.8-py3-none-any.whl.metadata (5.0 kB)
  Using cached langchain-1.2.7-py3-none-any.whl.metadata (4.9 kB)
  Using c

### Setting Up Sample Files

This notebook uses sample Word and PDF files.

When running the notebook on Google Colab, uncomment the code below to download the `datasets` directory from the Github repo.

In [2]:
import sys

IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    !git clone --no-checkout https://github.com/polzerdo55862/RAG-with-Python-Cookbook.git
    %cd RAG-with-Python-Cookbook
    !git sparse-checkout init --cone
    !git sparse-checkout set datasets
    !git checkout
    !cp -r datasets /content/datasets
    print("\u2713 Datasets downloaded to /content/datasets")
else:
    print("\u26a0 Running locally. Using ../datasets/ directory")

Cloning into 'RAG-with-Python-Cookbook'...
remote: Enumerating objects: 1641, done.
remote: Counting objects: 100% (382/382), done.
remote: Compressing objects: 100% (188/188), done.
remote: Total 1641 (delta 253), reused 249 (delta 190), pack-reused 1259 (from 2)
Receiving objects: 100% (1641/1641), 42.80 MiB | 8.58 MiB/s, done.
Resolving deltas: 100% (937/937), done.
/content/RAG-with-Python-Cookbook
Your branch is up to date with 'origin/main'.
✓ Datasets downloaded to /content/datasets


### Setting Up API Secrets

If you run this code in Google Colab, save your OpenAI API key in the colab secrets and load it to the environmental variables.

In [4]:
import os
import sys
from dotenv import load_dotenv

# Check if running in Google Colab
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    try:
        from google.colab import userdata  # type: ignore

        os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
        os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
        os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
    except ModuleNotFoundError:
        pass
else:
    load_dotenv()

## Metadata Extraction and Filtering

In [7]:
import PyPDF2
import os

file_path = "../datasets/pdf_files/attention_is_all_you_need_paper.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)
    metadata = reader.metadata

    text = ""
    for page in reader.pages:
        text += page.extract_text()
metadata


{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False'}

In [9]:
metadata_ext = dict(metadata)
metadata_ext["page_count"] = len(reader.pages)
metadata_ext["file_size"] = os.path.getsize(file_path)
metadata_ext["file_name"] = os.path.basename(file_path)
metadata_ext["file_path"] = file_path
metadata_ext["text_length"] = len(text)
metadata_ext


{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False',
 'page_count': 15,
 'file_size': 2215244,
 'file_name': 'attention_is_all_you_need_paper.pdf',
 'file_path': '../datasets/pdf_files/attention_is_all_you_need_paper.pdf',
 'text_length': 39472}

In [11]:
from pydantic import BaseModel
from openai import OpenAI

client = OpenAI()

class AuthorContact(BaseModel):
    name: str
    company: str
    email: list[str]

class Contacts(BaseModel):
    entries: list[AuthorContact]

system_message = """Extract the contact information of all authors."""

response = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[
        {
            "role": "system",
            "content": system_message,
        },
        {
            "role": "user",
            "content": text,
        },
    ],
    response_format=Contacts,
)

author_contacts = response.choices[0].message.parsed

metadata_ext_llm = dict(metadata_ext)
metadata_ext_llm["author_contacts"] = author_contacts
metadata_ext_llm


{'/Author': '',
 '/CreationDate': 'D:20240410211143Z',
 '/Creator': 'LaTeX with hyperref',
 '/Keywords': '',
 '/ModDate': 'D:20240410211143Z',
 '/PTEX.Fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
 '/Producer': 'pdfTeX-1.40.25',
 '/Subject': '',
 '/Title': '',
 '/Trapped': '/False',
 'page_count': 15,
 'file_size': 2215244,
 'file_name': 'attention_is_all_you_need_paper.pdf',
 'file_path': '../datasets/pdf_files/attention_is_all_you_need_paper.pdf',
 'text_length': 39472,
 'author_contacts': Contacts(entries=[AuthorContact(name='Ashish Vaswani', company='Google Brain', email=['avaswani@google.com']), AuthorContact(name='Noam Shazeer', company='Google Brain', email=['noam@google.com']), AuthorContact(name='Niki Parmar', company='Google Research', email=['nikip@google.com']), AuthorContact(name='Jakob Uszkoreit', company='Google Research', email=['usz@google.com']), AuthorContact(name='Llion Jones', company='Google Research', email

## Data Quality Enhancement: Abbreviation Expansion

In [13]:
import re

abbreviations_dict = {
    "NLP": "Natural Language Processing",
    "RNN": "Recurrent Neural Network",
    "LSTM": "Long Short-Term Memory",
    "GRU": "Gated Recurrent Unit",
    "TF": "Transformer",
    "MHA": "Multi-Head Attention",
    "FFN": "Feed-Forward Network",
}

file_path = "../datasets/text_files/blog_post_transformers.txt"
with open(file_path, "r") as file:
    text = file.read()

# Replace abbreviations in the text
for abbr, full_form in abbreviations_dict.items():
    text = re.sub(rf"\b{abbr}\b", f"{full_form} ({abbr})", text)

text


'Unveiling Transformer Models: A Revolution in ML\n\nIn the realm of ML and DL, few advancements have reshaped the field as profoundly as Transformer models. First introduced in the seminal paper "Attention Is All You Need," Transformer models have become the backbone of modern Natural Language Processing (NLP) and are extending their reach into CV and beyond.\n\nThe Challenges Before Transformers\n\nPrior to Transformers, models like RNNs, LSTMs, and GRUs were the primary tools for sequential data. While effective, these architectures faced significant challenges:\n\nSequential Computation: Processing one step at a time limited their ability to leverage parallelism, making training slower.\n\nLong-Range Dependencies: Understanding relationships between distant elements in a sequence was difficult.\n\nVanishing Gradients: Gradients diminished over long sequences, hampering effective learning.\n\nEnter Transformers, which bypass these limitations with a novel approach: SA.\n\nThe Anatom

In [14]:
import os
from openai import OpenAI

file_path = "../datasets/text_files/EMEA_drives_revenue.txt"

with open(file_path, "r") as file:
    text = file.read()

prompt = f"""
    The text below contains a financial report including a lot of
    abbreviations and technical terms from the finance domain.
    Please replace the abbreviations with their full forms and
    provide a brief explanation of the technical terms, so the
    whole text gets easier to read and understandable for everyone.

    Make sure it's easy enough, so that a 10-year-old school kid could
    understand it.

    Often used abbreviations are:
    - EMEA: Europe, Middle East, and Africa
    - BD: Business Development
    - YoY: Year-over-Year
    - APAC: Asia-Pacific

    Text:
    {text}
    """.strip()

client = OpenAI()
chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": prompt,
        }
    ],
    model="gpt-5.2",
)

enhanced_text = chat_completion.choices[0].message.content
enhanced_text


'**Europe, Middle East, and Africa (EMEA) Drives Big Sales Jump in Quarter 2 of 2019 (Q2 2019): Business Development (BD) Team Delivers Strong Year-over-Year (YoY) Growth**\n\nIn **Quarter 2 of 2019 (Q2 2019)**, the **Business Development (BD)** team helped the company sell a lot more than before. The **Europe, Middle East, and Africa (EMEA)** region was the biggest reason for this growth. Compared with the same time last year, sales grew by as much as **2.3x** (that means **about 2.3 times as much**, or **more than double**).\n\n---\n\n## Key Highlights (in simpler words)\n\n- **Europe, Middle East, and Africa (EMEA) was strongest:**  \n  Sales in **20 different countries** across EMEA made Quarter 2 of 2019 very successful.\n\n- **Top contributing countries:**  \n  The countries that added the most to total sales in Quarter 2 of 2019 were:  \n  - **United States (US): 28%**  \n  - **Spain: 20%**  \n  - **India: 15%**  \n  - **Norway: 11%**\n\n- **Year-over-Year (YoY) growth:**  \n  T

In [15]:
# write enhanced_text to a new .txt file
output_file_path = "../datasets/text_files/EMEA_drives_revenue_enhanced.txt"
with open(output_file_path, "w") as file:
    file.write(enhanced_text)

## Hypothetical Question Embedding for Enhanced Retrieval

In [17]:
import PyPDF2

file_path = "../datasets/pdf_files/AI_in_Factories_Discussion_Cleaned.pdf"

with open(file_path, "rb") as file:
    # Create PDF reader object
    reader = PyPDF2.PdfReader(file)

    # Extract text from all pages
    text = ""
    for page in reader.pages:
        text += page.extract_text()

text


"Discussion: Pros and Cons of AI in Factories\nAlex: Hey Sam, I read this article about AI revolutionizing factory work. It sounds amazing! Did you \nknow that AI-powered predictive maintenance can reduce downtime by up to 30%? Imagine how\nmuch that could improve productivity.\nSam: Yeah, I've read about that too. AI definitely has potential, but I feel like there's more to \nconsider than just efficiency. For instance, one study found that automation and AI could displace \nup to 20 million manufacturing jobs worldwide by 2030.\nAlex: That's a fair point, but couldn't companies invest in reskilling programs? According to a report \nby the World Economic Forum, 50% of all employees will need reskilling by 2025 due to the\nadoption of \nnew technologies. It's not impossible-it's about prioritizing it.\nSam: True, but not every company has the resources to reskill workers effectively. And there's also \nthe question of cost. AI implementation is expensive-customized AI solutions for fac

In [19]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    separators=[
        "\n\n",
    ],
)

text_chunks = text_splitter.split_text(text)
text_chunks

["Discussion: Pros and Cons of AI in Factories\nAlex: Hey Sam, I read this article about AI revolutionizing factory work. It sounds amazing! Did you \nknow that AI-powered predictive maintenance can reduce downtime by up to 30%? Imagine how\nmuch that could improve productivity.\nSam: Yeah, I've read about that too. AI definitely has potential, but I feel like there's more to \nconsider than just efficiency. For instance, one study found that automation and AI could displace \nup to 20 million manufacturing jobs worldwide by 2030.\nAlex: That's a fair point, but couldn't companies invest in reskilling programs? According to a report \nby the World Economic Forum, 50% of all employees will need reskilling by 2025 due to the\nadoption of \nnew technologies. It's not impossible-it's about prioritizing it.\nSam: True, but not every company has the resources to reskill workers effectively. And there's also \nthe question of cost. AI implementation is expensive-customized AI solutions for fa

In [20]:
import textwrap
from openai import OpenAI
from pydantic import BaseModel

file_path = "../datasets/text_files/AI_in_factories_chat.txt"

with open(file_path, "r", encoding="utf-8") as file:
    text = file.read()

client = OpenAI()

prompt = textwrap.dedent(
    f"""
    Below you can find a chat history between two students.

    Please generate 5 hypothetical questions that could be
    answered using the information from the discussion.
    The questions should focus on key details, definitions, and
    information present in the text.

    Chat History:
    {text}
    """
)

class HypotheticalQuestions(BaseModel):
    questions: list[str]

result = client.beta.chat.completions.parse(
    model="gpt-5-mini",
    messages=[{"role": "user", "content": prompt}],
    response_format=HypotheticalQuestions,
)

hypothetical_questions = result.choices[0].message.parsed.questions
hypothetical_questions


["How would implementing RPA that can reduce manufacturing costs by as much as 20% affect a factory's operating budget and competitiveness?",
 'What safety improvements might be expected if AI took over hazardous tasks, given that nearly 3 million workplace injuries occurred in the U.S. in 2022 with many in manufacturing?',
 'If machine vision systems can achieve 99% accuracy in quality control inspections, how would defect detection rates and product quality compare to relying on human inspectors?',
 'What cybersecurity measures would factories need to implement in response to an 87% increase in cyberattacks on industrial systems over the past two years?',
 'How could cloud-based AI tools enable smaller factories to access advanced analytics and help mitigate the inequality risks of mass AI adoption by larger corporations?']

## Character-Based Document Splitting

In [21]:
file_path = "../datasets/text_files/blog_post_transformers.txt"

with open(file_path, "r") as file:
    text = file.read()

def split_by_characters(text, chunk_size, overlap):
    chunks = []
    step = max(1, chunk_size - overlap)

    for start in range(0, len(text), step):
        end = start + chunk_size
        chunk = text[start:end]
        if chunk:
            chunks.append(chunk)

    return chunks

chunks = split_by_characters(text, chunk_size=100, overlap=20)
chunks


['Unveiling Transformer Models: A Revolution in ML\n\nIn the realm of ML and DL, few advancements have r',
 ' advancements have reshaped the field as profoundly as Transformer models. First introduced in the s',
 ' introduced in the seminal paper "Attention Is All You Need," Transformer models have become the bac',
 ' have become the backbone of modern NLP and are extending their reach into CV and beyond.\n\nThe Chall',
 'd beyond.\n\nThe Challenges Before Transformers\n\nPrior to Transformers, models like RNNs, LSTMs, and G',
 'e RNNs, LSTMs, and GRUs were the primary tools for sequential data. While effective, these architect',
 'ive, these architectures faced significant challenges:\n\nSequential Computation: Processing one step ',
 'Processing one step at a time limited their ability to leverage parallelism, making training slower.',
 'ing training slower.\n\nLong-Range Dependencies: Understanding relationships between distant elements ',
 'en distant elements in a sequence was dif

## Recursive Text Splitting Methods

In [ ]:

from langchain_text_splitters import RecursiveCharacterTextSplitter
import PyPDF2

file_path = "../datasets/pdf_files/daily_insights.pdf"

with open(file_path, "rb") as file:
    reader = PyPDF2.PdfReader(file)

    text = ""
    for page in reader.pages:
        text += page.extract_text()

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=200,
    chunk_overlap=0,
    length_function=len,
    is_separator_regex=False,
)

chunks = text_splitter.split_text(text)
chunks


## Document-Aware Splitting

In [ ]:

import os
from langchain_text_splitters import (
    PythonCodeTextSplitter,
    LatexTextSplitter,
    MarkdownHeaderTextSplitter,
)

file_path = "../datasets/markdown_files/random_md_code.md"
file_extension = os.path.splitext(file_path)[1]

with open(file_path, "r") as file:
    file_text = file.read()

if file_extension == ".py":
    splitter = PythonCodeTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".tex":
    splitter = LatexTextSplitter(chunk_size=500, chunk_overlap=50)
elif file_extension == ".md":
    headers_to_split_on = [
        ("#", "Header 1"),
        ("##", "Header 2"),
        ("###", "Header 3"),
    ]

    splitter = MarkdownHeaderTextSplitter(headers_to_split_on)

chunks = splitter.split_text(file_text)
chunks

## Semantic-Aware Text Chunking

In [ ]:
from langchain_experimental.text_splitter import SemanticChunker
from langchain_openai.embeddings import OpenAIEmbeddings

file_path = (
    "../datasets/text_files/"
    "random-text-about-5-different-stories.txt"
)

with open(file_path, "r") as file:
    text = file.read()

text_splitter = SemanticChunker(
    OpenAIEmbeddings(),
    breakpoint_threshold_type="percentile",
    breakpoint_threshold_amount=90,
)

chunks = text_splitter.split_text(text)


In [ ]:
chunks

## Agentic Text Chunking

In [ ]:
from langchain import hub
from langchain_openai import ChatOpenAI
from pydantic import BaseModel
from typing import List

# pull the prompt template from the langchain hub
obj = hub.pull("wfh/proposal-indexing")

llm = ChatOpenAI(model="gpt-5.2")

class Sentences(BaseModel):
    sentences: List[str]

extraction_llm = llm.with_structured_output(Sentences)

# Create the sentence extraction chain
extraction_chain = obj | extraction_llm

propositions = extraction_chain.invoke(
    """
    On July 20, 1969, astronaut Neil Armstrong walked on the moon .
    He was leading the NASA's Apollo 11 mission.
    Armstrong famously said, "That's one small step for man, one
    giant leap for mankind" as he stepped onto the lunar surface.
    """
)


In [ ]:
for sentence in propositions.sentences:
    print(sentence)